# Verificación de SHAP en los modelos finales de ML

Este cuaderno comprueba si **el valor de referencia más la suma de las contribuciones reproduce la salida del modelo** para RL, RF y XGBoost en México, España y Cuba, con la semilla 42.

## Cómo utilizarlo

1. Coloca este archivo en la carpeta `notebooks` del proyecto, en el equipo que guarda los modelos finales originales.
2. Selecciona el mismo entorno que utilizaste para ejecutar el notebook 02.
3. Ejecuta las celdas de este cuaderno en orden. Puedes usar «Ejecutar todo»: aquí no hay búsqueda de hiperparámetros ni entrenamiento.
4. Conserva el informe que se genera en `data/resultados_ml_groupcv_tfidf_lr_rf_10k/verificacion_shap/`.

Los modelos se cargan desde `modelos_finales`. Si faltan, el cuaderno se detiene: **no los sustituye por modelos reentrenados**. Tampoco modifica los modelos, los CSV de SHAP publicados ni las figuras existentes. Cada ejecución crea una carpeta nueva para su informe.

El filtro del test conserva los mismos IDs que el notebook 02 en los archivos revisados: 598 textos de México, 577 de España y 599 de Cuba. Las exclusiones están registradas en la celda de configuración y en el informe.

La tolerancia inicial sigue el criterio propuesto en la revisión, 0.001. También se registran los errores reales y cuántos superan 0.000001 y 0.00001 para revisar la precisión sin ocultarla tras el resultado «PASA».


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import importlib.metadata
import json
import platform
import sys
import traceback
import warnings

# Dejar None si este notebook está dentro de la carpeta del proyecto.
# Si no encuentra los datos, indicar aquí la carpeta que contiene data y notebooks.
RAIZ_PROYECTO = None
SEMILLA = 42
ATOL = 1e-3
RTOL = 0.0
TAMANO_LOTE = 32
N_REFERENCIA_RF = 100
VARIANTES = ["mx", "es", "cu"]
MODELOS = ["LR", "RF", "XGBoost"]
SUFIJOS = {"normal": "", "stem": "_stem", "lemma": "_lemma"}
RASGOS = ["n_exc", "n_int", "n_may", "n_emo", "n_ris", "n_neg", "n_elo", "n_com", "n_pun"]
TEST_ESPERADO = {"mx": 598, "es": 577, "cu": 599}
# IDs excluidos por el mismo filtro del notebook 02 en los archivos revisados.
IDS_EXCLUIDOS = {'mx': ['61a69eb1aa7c10192d9b2e7c06927ab2', '910b1a08a43e051796edf541480ea2ae'], 'es': ['002dec9f773cc17edcc7b02acb8e8926', '155de908c40fd05c363fcaf60dca1e82', '21235260c927996db9dc6540a0b7afe3', '239c97d7f321f824177f192b80d2ed41', '24c82035ece7175b69f93e301ca2b59b', '255d54507452bd211f8ff8dad1dc995e', '2d757c2ac5ac4b9c4a25f58474265198', '3012235006a1d11ccdab184cc9140bf1', '3e1a7f08d81abff11090c66562e453a0', '41328665c6181ca9d0a8a7305b83aa7d', '475cd8c3224669f377f7cb3269d4c356', '4e5b389ed6f1087234e78ea772c7c393', '55e6560a8467041ae7d98b3c75a09703', '608c3935b35a7e1263fd7cafe0a3a009', '79d003cb61c743b26959d523695c24d8', '7bcdbbe93f18129e7cbf231487abfe30', '807327f4adead1d48fc2317a1f70588b', '85eeafe1fe4b2c97103a123586868893', '8f9995ca9cc2ab7da7e82b24ac2cd28a', '97bb1718fc19f5236da23f1718139dd5', 'b3623b2dd85f09e3e9e41d9a9f570a95', 'c5c26b10a55814f64445d55aa6d05490', 'e633d3f337058ad661678112dfe6b0fe'], 'cu': ['eee65a05f91f621935e693d6152fad97']}

candidatos = [Path(RAIZ_PROYECTO)] if RAIZ_PROYECTO else [Path.cwd(), *Path.cwd().parents]
raices = [r for r in candidatos if (r / "data/resultados_ml_groupcv_tfidf_lr_rf_10k/seleccion_final_ml.csv").is_file()]
if not raices:
    raise FileNotFoundError("No encuentro el proyecto. Coloca este cuaderno en notebooks o indica RAIZ_PROYECTO.")
RAIZ = raices[0].resolve()
DATA = RAIZ / "data"
RESULTADOS = DATA / "resultados_ml_groupcv_tfidf_lr_rf_10k"
CARPETA_MODELOS = RESULTADOS / "modelos_finales"
if not CARPETA_MODELOS.is_dir():
    raise FileNotFoundError(
        f"Faltan los modelos entrenados: {CARPETA_MODELOS}. "
        "Ejecuta esta comprobación donde estén los archivos originales. Este cuaderno no los reentrena."
    )

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.exceptions import InconsistentVersionWarning
from IPython.display import display

seleccion = pd.read_csv(RESULTADOS / "seleccion_final_ml.csv")
if len(seleccion) != 9 or set(zip(seleccion.variante, seleccion.modelo)) != {
    (v, m) for v in VARIANTES for m in MODELOS
}:
    raise ValueError("La tabla de selección no contiene exactamente los nueve modelos esperados.")

def ruta_modelo(fila):
    return CARPETA_MODELOS / f"{fila.modelo.lower()}_{fila.variante}_{fila.preprocessing}_seed{SEMILLA}.pkl"

faltantes = [str(ruta_modelo(f)) for f in seleccion.itertuples() if not ruta_modelo(f).is_file()]
if faltantes:
    raise FileNotFoundError("Faltan estos modelos originales:\n" + "\n".join(faltantes))

# No instala ni actualiza librerías. Usar el entorno de la ejecución original.
for paquete in ["shap", "xgboost", "scikit-learn", "numpy", "pandas", "scipy", "joblib"]:
    try:
        importlib.metadata.version(paquete)
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(f"Falta {paquete}. Selecciona el entorno original del notebook 02.") from exc

marca = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
SALIDA = DATA / "resultados_interpretabilidad" / "validacion_ml" / "nueva_ejecucion" / marca
SALIDA.mkdir(parents=True, exist_ok=False)
versiones = {p: importlib.metadata.version(p) for p in [
    "shap", "xgboost", "scikit-learn", "numpy", "pandas", "scipy", "joblib"
]}
manifest = {
    "fecha_utc": marca, "python": sys.version, "plataforma": platform.platform(),
    "versiones": versiones, "semilla": SEMILLA, "atol": ATOL, "rtol": RTOL,
    "n_referencia_rf": N_REFERENCIA_RF, "ids_excluidos": IDS_EXCLUIDOS,
    "alcance": "Reconstrucción de la salida de nueve modelos finales originales, sin entrenamiento.",
    "archivos": {},
}

def huella(path):
    path = Path(path)
    if str(path) not in manifest["archivos"]:
        digest = hashlib.sha256()
        with path.open("rb") as fh:
            for bloque in iter(lambda: fh.read(1024 * 1024), b""):
                digest.update(bloque)
        manifest["archivos"][str(path)] = digest.hexdigest()
    return manifest["archivos"][str(path)]

huella(RESULTADOS / "seleccion_final_ml.csv")
(SALIDA / "entorno_y_procedencia.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Modelos localizados:", len(seleccion))
print("Versiones:", versiones)
print("Resultados de esta comprobación:", SALIDA)


## Cálculo y comprobación

Estas funciones reproducen los métodos de la sección 11 del notebook 02 y añaden la reconstrucción de la salida.


In [ ]:
def cargar_datos(variante, prep, split):
    path = DATA / f"{split}_clean{SUFIJOS[prep]}_{variante}.csv"
    huella(path)
    df = pd.read_csv(path, dtype={"ID": "string"}).reset_index(drop=True)
    if df["ID"].isna().any() or df["ID"].duplicated().any():
        raise ValueError(f"IDs ausentes o repetidos en {path.name}")
    faltan = [c for c in ["MESSAGE_CLEAN", *RASGOS] if c not in df]
    if faltan:
        raise ValueError(f"Faltan columnas en {path.name}: {faltan}")
    X = df[["MESSAGE_CLEAN", *RASGOS]].copy()
    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[RASGOS] = X[RASGOS].fillna(0)
    return df, X


def verificar_parametros(estimador, fila):
    paso = "xgb" if fila.modelo == "XGBoost" else "clf"
    actuales = estimador.get_params(deep=True)
    esperados = ast.literal_eval(fila.best_params)
    for nombre, esperado in esperados.items():
        clave = nombre if "__" in nombre else f"{paso}__{nombre}"
        if clave not in actuales:
            raise ValueError(f"No encuentro el parámetro guardado {clave}")
        actual = actuales[clave]
        if isinstance(esperado, (int, float)) and not isinstance(esperado, bool):
            coincide = actual is not None and np.isclose(actual, esperado, rtol=1e-12, atol=1e-15)
        else:
            coincide = actual == esperado
        if not coincide:
            raise ValueError(f"El modelo no coincide con la selección final: {clave}: {actual!r} != {esperado!r}")
    clf = estimador.named_steps[paso]
    if clf.get_params().get("random_state") != SEMILLA:
        raise ValueError("La semilla del modelo no coincide con 42.")
    if np.asarray(clf.classes_).tolist() != [0, 1]:
        raise ValueError("Se esperaban las clases [0, 1], con 1 = irónico.")


def preparar_calculo(modelo, estimador, X_train):
    preproc = estimador.named_steps["features"]
    clf = estimador.named_steps["xgb" if modelo == "XGBoost" else "clf"]
    contexto = {"modelo": modelo, "preproc": preproc, "clf": clf}
    if modelo == "LR":
        Mtr = sp.csr_matrix(preproc.transform(X_train), dtype=np.float64)
        media = np.asarray(Mtr.mean(axis=0)).ravel()
        coef = np.asarray(clf.coef_, dtype=np.float64).ravel()
        contexto.update(coef=coef, media=media,
                        base=float(media @ coef + np.asarray(clf.intercept_).ravel()[0]),
                        escala="log-odds", modo="lineal_independiente")
    elif modelo == "XGBoost":
        contexto.update(escala="margen_log-odds", modo="TreeSHAP_nativo")
    elif modelo == "RF":
        import shap
        Mtr = sp.csr_matrix(preproc.transform(X_train), dtype=np.float64)
        if Mtr.shape[0] < N_REFERENCIA_RF:
            raise ValueError("El entrenamiento contiene menos de 100 textos de referencia.")
        rng = np.random.RandomState(SEMILLA)
        indices = rng.choice(Mtr.shape[0], N_REFERENCIA_RF, replace=False)
        explicador = shap.TreeExplainer(
            clf, data=Mtr[indices].toarray(),
            feature_perturbation="interventional", model_output="probability"
        )
        contexto.update(explicador=explicador, indices_fondo=indices,
                        escala="probabilidad_clase_1", modo="TreeSHAP_interventional")
    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")
    return contexto


def calcular_bloque(contexto, X):
    modelo, clf = contexto["modelo"], contexto["clf"]
    M = sp.csr_matrix(contexto["preproc"].transform(X), dtype=np.float64)
    if modelo == "LR":
        valores = (M.toarray() - contexto["media"]) * contexto["coef"]
        base = np.full(M.shape[0], contexto["base"], dtype=np.float64)
        salida = np.asarray(clf.decision_function(M), dtype=np.float64).reshape(-1)
    elif modelo == "XGBoost":
        import xgboost as xgb
        dm = xgb.DMatrix(M)
        bruto = np.asarray(clf.get_booster().predict(
            dm, pred_contribs=True, validate_features=False
        ), dtype=np.float64)
        if bruto.ndim != 2 or bruto.shape[1] != M.shape[1] + 1:
            raise ValueError(f"Dimensiones inesperadas de TreeSHAP: {bruto.shape}")
        valores, base = bruto[:, :-1], bruto[:, -1]
        salida = np.asarray(clf.get_booster().predict(
            dm, output_margin=True, validate_features=False
        ), dtype=np.float64).reshape(-1)
    else:
        explicador = contexto["explicador"]
        # La comprobación independiente de abajo compara TODAS las filas con
        # predict_proba. No dependemos del chequeo interno de la biblioteca.
        bruto = explicador.shap_values(M.toarray(), check_additivity=False)
        esperados = np.asarray(explicador.expected_value, dtype=np.float64).ravel()
        if isinstance(bruto, list):
            valores = np.asarray(bruto[1], dtype=np.float64)
            referencia = esperados[1]
        else:
            bruto = np.asarray(bruto, dtype=np.float64)
            if bruto.ndim == 3 and bruto.shape[2] == 2 and esperados.size == 2:
                valores, referencia = bruto[:, :, 1], esperados[1]
            elif bruto.ndim == 2 and esperados.size == 1:
                valores, referencia = bruto, esperados[0]
            else:
                raise ValueError(f"Dimensiones inesperadas de SHAP para RF: {bruto.shape}; base={esperados.shape}")
        base = np.full(M.shape[0], referencia, dtype=np.float64)
        salida = np.asarray(clf.predict_proba(M)[:, 1], dtype=np.float64)
    if valores.shape != M.shape or base.shape != salida.shape or len(salida) != M.shape[0]:
        raise ValueError("Las contribuciones y la salida no están alineadas.")
    return base, valores, salida


In [ ]:
registros = []
for fila in seleccion.sort_values(["variante", "modelo"]).itertuples():
    etiqueta = f"{fila.variante.upper()} / {fila.modelo}"
    resumen = {"variante": fila.variante, "modelo": fila.modelo,
               "preprocessing": fila.preprocessing, "semilla": SEMILLA,
               "atol": ATOL, "rtol": RTOL, "estado": "ERROR"}
    print(f"\nIniciando {etiqueta}...", flush=True)
    try:
        path = ruta_modelo(fila)
        resumen["sha256_modelo"] = huella(path)
        with warnings.catch_warnings(record=True) as avisos:
            warnings.simplefilter("always")
            estimador = joblib.load(path)
        mensajes = [str(w.message) for w in avisos]
        resumen["avisos_carga"] = " | ".join(mensajes)
        if any(issubclass(w.category, InconsistentVersionWarning) for w in avisos):
            raise RuntimeError("La versión de scikit-learn no coincide con la del modelo. Usar el entorno original.")
        verificar_parametros(estimador, fila)
        df_train, X_train = cargar_datos(fila.variante, fila.preprocessing, "train")
        df_test, X_test = cargar_datos(fila.variante, fila.preprocessing, "test")
        if len(df_test) != 600:
            raise ValueError("Se esperaban 600 textos del test oficial antes del filtro.")
        excluidos = set(IDS_EXCLUIDOS[fila.variante])
        if not excluidos.issubset(set(df_test["ID"].astype(str))):
            raise ValueError("Los IDs del test no coinciden con el filtro de los archivos revisados.")
        conservar = ~df_test["ID"].astype(str).isin(excluidos)
        df_test = df_test.loc[conservar].reset_index(drop=True)
        X_test = X_test.loc[conservar].reset_index(drop=True)
        if len(df_test) != TEST_ESPERADO[fila.variante]:
            raise ValueError("El tamaño del test filtrado no coincide con el usado en el estudio.")
        contexto = preparar_calculo(fila.modelo, estimador, X_train)
        resumen.update(n_train=len(df_train), n_test=len(df_test),
                       escala=contexto["escala"], modo=contexto["modo"])
        if fila.modelo == "RF":
            ids_fondo = df_train.iloc[contexto["indices_fondo"]][["ID"]]
            ids_fondo.to_csv(SALIDA / f"referencia_RF_{fila.variante}.csv", index=False)
        detalles = []
        for inicio in range(0, len(X_test), TAMANO_LOTE):
            fin = min(inicio + TAMANO_LOTE, len(X_test))
            base, valores, salida = calcular_bloque(contexto, X_test.iloc[inicio:fin])
            reconstruida = base + valores.sum(axis=1, dtype=np.float64)
            error = np.abs(reconstruida - salida)
            finitos = (np.isfinite(valores).all(axis=1) & np.isfinite(base)
                       & np.isfinite(salida) & np.isfinite(reconstruida))
            tolerancia = ATOL + RTOL * np.abs(salida)
            cumple = finitos & (error <= tolerancia)
            detalles.append(pd.DataFrame({
                "ID": df_test["ID"].iloc[inicio:fin].astype(str).to_numpy(),
                "variante": fila.variante, "modelo": fila.modelo,
                "preprocessing": fila.preprocessing, "semilla": SEMILLA,
                "escala": contexto["escala"], "valor_referencia": base,
                "suma_contribuciones": valores.sum(axis=1, dtype=np.float64),
                "salida_modelo": salida, "salida_reconstruida": reconstruida,
                "error_absoluto": error, "tolerancia": tolerancia,
                "valores_finitos": finitos, "cumple": cumple,
            }))
            print(f"  {etiqueta}: {fin}/{len(X_test)} textos", flush=True)
        detalle = pd.concat(detalles, ignore_index=True)
        detalle.to_csv(SALIDA / f"detalle_{fila.modelo}_{fila.variante}.csv", index=False)
        errores = detalle.loc[~detalle["cumple"]]
        errores.to_csv(SALIDA / f"incumplimientos_{fila.modelo}_{fila.variante}.csv", index=False)
        finitos = detalle["valores_finitos"].all()
        resumen.update(
            n_incumplimientos=int(len(errores)),
            n_no_finitos=int((~detalle["valores_finitos"]).sum()),
            error_maximo=float(detalle["error_absoluto"].max()) if finitos else float("inf"),
            error_medio=float(detalle["error_absoluto"].mean()) if finitos else float("inf"),
            n_error_mayor_1e_6=int((detalle["error_absoluto"] > 1e-6).sum()),
            n_error_mayor_1e_5=int((detalle["error_absoluto"] > 1e-5).sum()),
            n_error_mayor_1e_3=int((detalle["error_absoluto"] > 1e-3).sum()),
            estado="PASA" if errores.empty else "NO_PASA",
        )
        print(f"  {etiqueta}: {resumen['estado']} | error máximo={resumen['error_maximo']:.3e}", flush=True)
    except Exception as exc:
        resumen["error_ejecucion"] = f"{type(exc).__name__}: {exc}"
        (SALIDA / f"error_{fila.modelo}_{fila.variante}.txt").write_text(traceback.format_exc(), encoding="utf-8")
        print(f"  {etiqueta}: ERROR — {exc}", flush=True)
    registros.append(resumen)
    pd.DataFrame(registros).to_csv(SALIDA / "resumen_verificacion_shap.csv", index=False)
    (SALIDA / "entorno_y_procedencia.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

resumen_final = pd.DataFrame(registros)
columnas = [c for c in ["variante", "modelo", "preprocessing", "n_test", "escala", "estado",
                        "error_maximo", "n_incumplimientos", "error_ejecucion"] if c in resumen_final]
display(resumen_final[columnas])
completo = len(resumen_final) == 9 and resumen_final["estado"].eq("PASA").all()
print("\nReconstrucción verificada para los nueve modelos:", "SÍ" if completo else "NO")
print("Carpeta con el informe:", SALIDA)


## Qué significa el resultado

- **PASA:** todas las filas reconstruyen la salida correspondiente dentro de la tolerancia configurada.
- **NO_PASA:** el cálculo terminó, pero hay filas que no cumplen esa condición o contienen valores no finitos.
- **ERROR:** no se pudo completar la comprobación. El informe guarda el motivo y el detalle del error; los demás modelos continúan.

Para revisar el resultado, comparte `resumen_verificacion_shap.csv` y `entorno_y_procedencia.json`. Si aparece `NO_PASA` o `ERROR`, conserva también los archivos de incumplimientos o errores correspondientes.

La comprobación verifica la consistencia de las contribuciones con la salida de **estos modelos originales**. No convierte las escalas en equivalentes, no demuestra que todos los supuestos de SHAP sean adecuados y no corrige por sí sola las diferencias entre raíces, lemas y palabras en las comparaciones de vocabulario.
